# Full-Quran multi-compressor consensus pipeline

**Pre-registered experiment**: `exp95e_full_114_consensus_universal` (H37)

**Hypothesis**: K = 2 consensus across {gzip-9, bz2-9, lzma-preset-9, zstd-9} with τ frozen from exp95c closes single-letter forgery detection on every Quran surah at aggregate recall ≥ 0.999 and per-surah recall ≥ 0.99.

## How to use this notebook

1. **Read Cell 1's scope-picker** and decide how long you want the run to take.
2. **Run cells in order** (1 → 11). Each cell prints a short summary so you can stop after any cell if a sentinel fires.
3. **Don't skip the sentinels**: τ-drift (Cell 2), corpus integrity (Cell 3), Q:100 regression (Cell 6) are gates. If any reports `ok=False`, abort and investigate before trusting the verdict.
4. **Cell 11 packages everything into a zip with a one-click download button.**

## Scope ladder (pick one in Cell 1)

The pre-registered **headline scope is `v1`** — first verse of every surah, all 27 substitutes per consonant. The verdict in PREREG §5 is computed on V1 only. Larger scopes (`short`, `sample`, `quarter`, `half`, `full`) are *diagnostic* — they receive a separate `consistency_verdict ∈ {CONSISTENT, INCONSISTENT_DIAGNOSTIC}` and cannot upgrade or downgrade the V1 verdict.

| Scope     | Variants  | Estimate (i7-7700HQ, 6 workers, AC power) | Use case                                     |
|-----------|----------:|-------------------------------------------|----------------------------------------------|
| `v1`      |  ~145 K   | **45 min – 1.5 h**                        | **DEFAULT — pre-registered headline**        |
| `short`   |  ~377 K   | 2 – 4 h                                   | First 3 verses per surah                     |
| `sample`  |  ~370 K   | 2 – 4 h                                   | Random 10 % of FULL                          |
| `quarter` |  ~925 K   | 5 – 8 h                                   | Random 25 % of FULL                          |
| `half`    | ~1.85 M   | 10 – 18 h                                 | Random 50 % of FULL (overnight)              |
| `full`    |  ~3.7 M   | 18 – 36 h                                 | Every interior letter (overnight + day)      |

## Honest scope reminders

- This pipeline tests *single-letter consonant substitutions* on the Hafs ʿan ʿĀṣim canonical text. Word-level edits, semantic substitutions, LLM-generated counterfeits, and cross-qirāʾāt confusions are **out of scope** (see `exp92_genai_adversarial_forge`).
- A PASS verdict is *single-team single-codebase* evidence. Community-grade "universal forgery tool" claim requires external two-team replication.
- There is no Shannon-capacity theorem here. This is empirical universal scaling, not a derivation.

## Cell 1 — Imports + **SCOPE picker** (CHANGE `SCOPE` HERE)

**This is the only configuration cell.** Set `SCOPE` to whichever runtime you want.

For your first run on a Legion Y720 (i7-7700HQ): leave `SCOPE = "v1"` and `WORKERS = 6`. That gives the headline pre-registered run in ~45–90 min on AC power.

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path
import numpy as np

# =================================================================
# SCOPE PICKER — change ONLY this string to switch scope.
# =================================================================
#   'v1'       ~145 K variants   ~45 min – 1.5 h    ← DEFAULT (headline)
#   'short'    ~377 K variants   ~2 – 4 h
#   'sample'   ~370 K variants   ~2 – 4 h            (random 10 % of full)
#   'quarter'  ~925 K variants   ~5 – 8 h            (random 25 % of full)
#   'half'    ~1.85 M variants   ~10 – 18 h          (random 50 % of full, overnight)
#   'full'    ~3.7 M variants    ~18 – 36 h          (every interior letter)
# =================================================================
SCOPE      = "v1"

# Worker pool tuning — for an i7-7700HQ (4 cores / 8 threads) leave 1–2 threads free.
WORKERS    = 6
CHUNK_SIZE = 64

# Optional: restrict to a subset of surahs (comma-separated labels). Set to None for all 114.
SURAHS_FILTER = None         # e.g. "Q:099,Q:100" for a quick 2-surah sanity run

# Locate the repo root regardless of where the notebook is launched from
_NB_DIR = Path.cwd()
_ROOT_CANDIDATES = [
    _NB_DIR, _NB_DIR.parent,
    Path(r"C:\Users\mtj_2\OneDrive\Desktop\Quran"),
]
_REPO_ROOT = next((p for p in _ROOT_CANDIDATES if (p / "experiments" / "_lib.py").exists()), None)
if _REPO_ROOT is None:
    raise SystemExit("Cannot locate the QSF repo root. Cd into the repo or edit _ROOT_CANDIDATES.")
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

# Import the experiment package — any syntax / import error surfaces here
from experiments.exp95e_full_114_consensus_universal import (
    _audit, _detector, _enumerate, run as exp95e_run,
)

# tqdm with notebook-aware fallback
try:
    from tqdm.auto import tqdm     # tqdm.auto chooses widget vs text automatically
    _HAS_TQDM = True
except ImportError:
    _HAS_TQDM = False
    def tqdm(it, **kw):              # graceful no-op fallback
        return it

# Headline echo so you can see the config you chose
print(f"Repo root      : {_REPO_ROOT}")
print(f"Scope          : {SCOPE!r}")
print(f"Workers        : {WORKERS}")
print(f"Chunk size     : {CHUNK_SIZE}")
print(f"Surahs filter  : {SURAHS_FILTER or 'ALL 114'}")
print(f"Python         : {sys.version.split()[0]}")
print(f"tqdm available : {_HAS_TQDM}")
assert SCOPE in ("v1", "short", "sample", "quarter", "half", "full"), \
    f"Unknown scope {SCOPE!r}; must be one of v1/short/sample/quarter/half/full"
print("\nAll exp95e modules imported successfully. Ready to run sentinels.")

## Cell 2 — τ-drift sentinel + receipt loading

Loads the locked τ values from the `exp95c` receipt and compares against the snapshot embedded in `run.py::_LOCKED_TAU`. If they differ by more than `1e-12`, **abort — DO NOT proceed**.

In [ ]:
exp95c = exp95e_run._load_exp95c_receipt()
exp94  = exp95e_run._load_exp94_receipt()

receipt_tau = {n: float(exp95c["tau_per_compressor"][n]) for n in exp95e_run.NAMES}
locked_tau  = exp95e_run._LOCKED_TAU

tau_drift = _audit.check_tau_drift(locked_tau, receipt_tau, tol=1e-12)
print("τ-drift sentinel")
print(f"  ok          : {tau_drift['ok']}")
print(f"  max drift   : {tau_drift['max_drift']:.2e}")
for n, info in tau_drift["per_compressor"].items():
    print(f"  {n:>5}: locked = {info['locked']:.18f}")
    print(f"         actual = {info['from_exp95c_receipt']:.18f}  ok = {info['ok']}")

if not tau_drift["ok"]:
    raise SystemExit("τ values drifted from exp95c receipt. Investigate before proceeding.")
print("\nReady to score.")

## Cell 3 — Corpus loading + integrity self-check begin

Loads `phase_06_phi_m.pkl` (SHA-256 verified by `_lib`), captures the protected-file hashes for the self-check end at the very last cell.

In [ ]:
from experiments._lib import load_phase, self_check_begin, safe_output_dir

pre = self_check_begin()
print(f"Self-check began at {pre['ts_start']} "
      f"({len(pre['files'])} protected files, "
      f"{sum(len(v) for v in pre['dirs'].values())} files in protected dirs).")

phi = load_phase("phase_06_phi_m")
CORPORA = phi["state"]["CORPORA"]
quran_all = list(CORPORA.get("quran", []))
if SURAHS_FILTER:
    wanted = {s.strip() for s in SURAHS_FILTER.split(",") if s.strip()}
    quran  = [u for u in quran_all if getattr(u, "label", "") in wanted]
else:
    quran = quran_all
print(f"\nCORPORA['quran']: {len(quran_all)} surahs")
print(f"After filter   : {len(quran)} surahs to score.")

out_root = safe_output_dir(exp95e_run.EXP)
out = out_root / SCOPE
out.mkdir(parents=True, exist_ok=True)
print(f"Output dir     : {out}")

## Cell 4 — Enumerate variants for the selected scope

Builds the chunked work-units for the worker pool and prints a per-surah variant-count summary so you can sanity-check enumeration BEFORE the long compute step.

In [ ]:
t_enum = time.time()
chunks = exp95e_run._build_chunks(quran, SCOPE, chunk_size=CHUNK_SIZE)
elapsed_enum = time.time() - t_enum

n_variants_per_surah = {}
for (label, _, pairs, _n_specs) in chunks:
    n_variants_per_surah[label] = n_variants_per_surah.get(label, 0) + len(pairs)

n_total = sum(n_variants_per_surah.values())
print(f"Enumeration took {elapsed_enum:.1f}s")
print(f"Total variants : {n_total:,}")
print(f"Surahs with ≥1 variant: {sum(1 for v in n_variants_per_surah.values() if v > 0)}")
print(f"\n  10 LARGEST surahs by variant count:")
for lbl, n in sorted(n_variants_per_surah.items(), key=lambda kv: -kv[1])[:10]:
    print(f"    {lbl}  n_variants = {n:>7,}")
print(f"\n  10 SMALLEST surahs (excluding zero):")
nonzero = [(l, n) for l, n in n_variants_per_surah.items() if n > 0]
for lbl, n in sorted(nonzero, key=lambda kv: kv[1])[:10]:
    print(f"    {lbl}  n_variants = {n:>7,}")

## Cell 5 — Parallel scoring loop with live progress bar

The heavy compute step. **This is where you wait.**

Time depends on scope and your CPU. Real-time runtime estimate updates appear in the tqdm progress bar (it shows: percentage, chunks done / total, elapsed, **estimated time to finish (ETA)**, and chunks/sec).

**Windows note** (patched 2026-04-26 night): `ProcessPoolExecutor` uses **spawn** on Windows, which means each worker starts a fresh Python interpreter and re-imports the module. The cell sets `os.environ["PYTHONPATH"]` to include the repo root *before* the pool is created — this makes the spawn re-imports succeed. **Without that env var, workers hang silently and the kernel restarts in a loop.** The previous `sys.path.insert` in Cell 1 only affects the parent kernel, not spawned workers.

**Expected first-task latency**: 10–30 s on Windows while each worker spawns a fresh Python interpreter and imports `numpy + zstandard + the worker module`. After that the bar fills steadily. **If the bar stays at 0 % for > 60 s** that's the spawn bug returning (kill kernel + verify `PYTHONPATH` line above ran successfully).

**Do not interrupt** unless you really need to stop — every chunk is independent so partial results are still valid, but the receipt won't be written.

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as _mp

# ---------------------------------------------------------------------
# Windows + Jupyter spawn-fix
# ---------------------------------------------------------------------
# On Windows, ProcessPoolExecutor uses *spawn*: every worker starts a
# fresh Python interpreter and re-imports the worker function's module.
# That re-import has to be able to find
# `experiments.exp95e_full_114_consensus_universal.run` on `sys.path`.
#
# Cell 1's runtime `sys.path.insert(0, _REPO_ROOT)` is NOT inherited by
# spawned workers — they get a clean interpreter. The cleanest reliable
# fix is to put the repo root on `PYTHONPATH` (which IS inherited via
# os.environ) before the pool is created. Without this, workers hang
# silently during import and the cell never makes progress (the symptom
# is "stuck at 0 % forever; kernel restart loop").
# ---------------------------------------------------------------------
os.environ["PYTHONPATH"] = (
    str(_REPO_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")
)
# Belt & suspenders: also pin the start method to "spawn" explicitly, in
# case some upstream code touched it. set_start_method can only be
# called once per process; suppress the inevitable RuntimeError on re-run.
try:
    _mp.set_start_method("spawn", force=False)
except RuntimeError:
    pass

score_inputs = []
for (label, canon_letters, pairs, _n_specs) in chunks:
    if pairs:
        score_inputs.append((label, canon_letters, pairs, receipt_tau))
n_chunks = len(score_inputs)
print(f"Dispatching {n_chunks:,} chunks ({n_total:,} variants) to {WORKERS} worker(s) ...")
print(f"  PYTHONPATH (head)     : {os.environ['PYTHONPATH'].split(os.pathsep)[0]}")
print(f"  mp start method       : {_mp.get_start_method()}")

t_score = time.time()
chunk_results = []
if WORKERS <= 1:
    bar = tqdm(score_inputs, desc=f"Scoring [{SCOPE}]", unit="chunk", smoothing=0.05)
    for ci in bar:
        chunk_results.append(exp95e_run._score_surah_chunk(ci))
else:
    # Explicit spawn context (matches Windows default; harmless on Linux/Mac).
    # First task may take ~10–30 s on Windows because each worker has to
    # spawn a fresh interpreter and import numpy + zstandard + the worker
    # module. Subsequent chunks are fast.
    spawn_ctx = _mp.get_context("spawn")

    with ProcessPoolExecutor(
        max_workers=WORKERS,
        mp_context=spawn_ctx,
    ) as ex:
        futures = [ex.submit(exp95e_run._score_surah_chunk, ci) for ci in score_inputs]
        bar = tqdm(
            as_completed(futures), total=len(futures),
            desc=f"Scoring [{SCOPE}]", unit="chunk", smoothing=0.05,
        )
        for fut in bar:
            chunk_results.append(fut.result())

elapsed_score = time.time() - t_score
print(f"\nScored {n_total:,} variants in {elapsed_score:.1f}s "
      f"({n_total / max(1.0, elapsed_score):.0f} variants/s).")

## Cell 6 — ctrl-null FPR re-computation + Q:100 regression check

Re-computes ctrl-null FPR using the same protocol as exp95c (200 ctrl units × 20 perturbations × 4 compressors). With locked τ this should agree with the exp95c receipt FPRs to within sampling noise.

Then verifies Q:100 inside this universal run reproduces:
- K = 2 recall = 1.000 (exp95c headline)
- gzip-solo recall = 0.990741 ± 0.001 (exp94 baseline)

The ctrl-null is single-threaded but small (~4000 variants × 4 compressors); a tqdm bar shows progress.

In [ ]:
# Per-surah aggregation from chunk_results
n_canon_v1 = {getattr(u, "label", "?"): len(_enumerate.letters_28(u.verses[0])) for u in quran}
per_surah = {
    getattr(u, "label", "?"): {
        "n_variants": 0,
        "fires_K_atleast": [0, 0, 0, 0, 0],
        "fires_solo": {n: 0 for n in exp95e_run.NAMES},
        "ncd_min": {n: float("inf") for n in exp95e_run.NAMES},
        "ncd_max": {n: float("-inf") for n in exp95e_run.NAMES},
        "missed": [],
        "n_canon_consonants_v1": n_canon_v1.get(getattr(u, "label", "?"), 0),
    }
    for u in quran
}
exp95e_run._aggregate(per_surah, chunk_results, n_canon_v1)
exp95e_run._per_surah_finalise(per_surah)
agg = exp95e_run._aggregate_global(per_surah)

# ctrl-null FPR — wrapped with tqdm via a thin re-implementation that mirrors
# the exp95c protocol byte-for-byte, but with progress visible.
import random as _random
print("Re-computing ctrl-null FPR with locked τ ...")
arabic_ctrl = ["poetry_jahili", "poetry_islami", "poetry_abbasi",
               "ksucca", "arabic_bible", "hindawi"]
lo, hi = 15, 100
pool = []
for name in arabic_ctrl:
    for u in CORPORA.get(name, []):
        nverses = len(u.verses)
        if lo <= nverses <= hi:
            pool.append(u)
rng_pool = _random.Random(42 + 1)
rng_pool.shuffle(pool)
units = pool[:200]
rng_c = _random.Random(42 + 2)

null_per = {n: [] for n in exp95e_run.NAMES}
bar = tqdm(units, desc="ctrl-null FPR", unit="unit", smoothing=0.05)
for u in bar:
    rng_u = _random.Random(rng_c.randrange(1 << 30))
    for _ in range(20):
        pair = exp95e_run._apply_ctrl_perturbation(u.verses, rng_u)
        if pair is None:
            continue
        pert_verses = pair[0]
        ca = _enumerate.letters_28(" ".join(u.verses)).encode("utf-8")
        cb = _enumerate.letters_28(" ".join(pert_verses)).encode("utf-8")
        cab = ca + cb
        for n in exp95e_run.NAMES:
            fn = exp95e_run._LEN_FNS[n]
            za, zb, zab = fn(ca), fn(cb), fn(cab)
            denom = max(1, max(za, zb))
            null_per[n].append((zab - min(za, zb)) / denom)

null_fires = {n: (np.asarray(null_per[n]) >= receipt_tau[n]).astype(int) for n in exp95e_run.NAMES}
null_K = sum(null_fires[n] for n in exp95e_run.NAMES)
ctrl_null = {
    "n_null_pool": len(null_per["gzip"]),
    "fpr_per_compressor_at_locked_tau": {n: float((np.asarray(null_per[n]) >= receipt_tau[n]).mean())
                                          for n in exp95e_run.NAMES},
    "fpr_by_consensus_K": {str(K): float((null_K >= K).mean()) for K in (1, 2, 3, 4)},
}
print("\nctrl-null FPR by consensus K:")
for K in (1, 2, 3, 4):
    print(f"  K={K}: {ctrl_null['fpr_by_consensus_K'][str(K)]:.4f}"
          f"   (target ≤ {exp95e_run.FPR_TARGET})")

print("\nQ:100 regression sub-result:")
q100_reg = _audit.check_q100_regression(
    per_surah, exp95e_run.EXP94_BASELINE_GZIP_RECALL,
    drift_tol=exp95e_run.PROTOCOL_DRIFT_TOL,
)
print(f"  K=2  expected = 1.000000  actual = {q100_reg['K2_recall_q100']['actual']:.6f}  ok = {q100_reg['K2_recall_q100']['ok']}")
print(f"  gzip expected = {exp95e_run.EXP94_BASELINE_GZIP_RECALL:.6f}  actual = {q100_reg['gzip_solo_recall_q100']['actual']:.6f}  ok = {q100_reg['gzip_solo_recall_q100']['ok']}")
print(f"  overall ok = {q100_reg['ok']}")

## Cell 7 — Per-surah aggregate, verdict, weakest-surah inspection

In [ ]:
# Variant-count sanity (V1 only)
if SCOPE == "v1":
    var_counts = _audit.check_variant_counts_v1(per_surah)
    if not var_counts["ok"]:
        print(f"WARNING: variant-count mismatch on {len(var_counts['mismatches'])} surahs")
        for m in var_counts['mismatches'][:5]:
            print(f"  {m}")
    else:
        print(f"variant-count sanity: ok ({var_counts['n_surahs']} surahs)")
else:
    var_counts = {"ok": True, "skipped_for_scope": SCOPE}
    print(f"variant-count sanity: skipped for scope={SCOPE}")

# Verdict
verdict = exp95e_run._verdict(SCOPE, tau_drift, q100_reg, ctrl_null,
                              per_surah, agg, exp95e_run.PROTOCOL_DRIFT_TOL)

print(f"\n{'=' * 68}")
print(f"PRE-REGISTERED VERDICT: {verdict}")
print(f"{'=' * 68}")
print(f"  scope                 : {SCOPE}")
print(f"  surahs with variants  : {sum(1 for r in per_surah.values() if r['n_variants'] > 0)} / {len(per_surah)}")
print(f"  total variants        : {agg['n_variants']:,}")
print(f"  aggregate recall K=1  : {agg['recall_K1']:.6f}")
print(f"  aggregate recall K=2  : {agg['recall_K2']:.6f}  ← headline")
print(f"  aggregate recall K=3  : {agg['recall_K3']:.6f}")
print(f"  aggregate recall K=4  : {agg['recall_K4']:.6f}")
print(f"  ctrl-null K=2 FPR     : {ctrl_null['fpr_by_consensus_K']['2']:.4f}")

# 10 weakest surahs
eligible = [(lbl, row) for lbl, row in per_surah.items() if row["n_variants"] > 0]
eligible.sort(key=lambda kv: kv[1]["recall_K2"])
print("\n10 WEAKEST surahs by K=2 recall:")
for lbl, row in eligible[:10]:
    print(f"  {lbl}  n={row['n_variants']:>5}  K1={row['recall_K1']:.6f}  K2={row['recall_K2']:.6f}  gzip-solo={row['recall_solo_gzip']:.6f}")

## Cell 8 — Missed-variant clustering

For each missed variant (`K_fired < 2`), cross-tabulate by (orig → repl) consonant pair to see whether the misses are concentrated in specific phonetic confusions (e.g. ب↔و from the Q:099 al-Zalzalah robustness study).

In [ ]:
all_missed = []
for row in per_surah.values():
    all_missed.extend(row.get("missed", []))

clusters = _audit.cluster_missed_variants(all_missed)
print(f"Total missed variants: {clusters['n_missed']}")
if clusters["n_missed"] == 0:
    print("  No misses → K=2 consensus closes every variant in this scope.")
else:
    print("\nTop missed-pair clusters (orig → repl, count):")
    for label, n in clusters["top_pairs"][:15]:
        print(f"  {label:>10}  n = {n}")
    print("\nMisses by source consonant:")
    for c, n in list(clusters["by_orig_consonant"].items())[:8]:
        print(f"  {c}  n = {n}")
    print("\nMisses by replacement consonant:")
    for c, n in list(clusters["by_replacement_consonant"].items())[:8]:
        print(f"  {c}  n = {n}")
    print("\nMisses by surah (top 10):")
    for lbl, n in list(clusters["by_surah"].items())[:10]:
        print(f"  {lbl}  n = {n}")

## Cell 9 — Deployable detector demo

Demonstrate `is_quranic()` on:

1. The canonical Q:100 → expect `AUTHENTIC` (K=0, exact-match short-circuit)
2. A hand-crafted single-letter forgery of Q:100 → expect `FORGED` (K≥2)
3. Q:001 passed under the Q:100 label (whole-surah mismatch) → expect `FORGED`

This is the actual deployable interface: same locked τ, same compressor primitives, no pickle dependency once `canon_text` is supplied directly.

In [ ]:
from experiments.exp95e_full_114_consensus_universal._detector import is_quranic, load_locked_tau

locked = load_locked_tau()
print("Locked τ (loaded from exp95c receipt):")
for n, t in locked.items():
    print(f"  {n:>5}: {t:.18f}")

q100 = next((u for u in CORPORA["quran"] if getattr(u, "label", "") == "Q:100"), None)
q001 = next((u for u in CORPORA["quran"] if getattr(u, "label", "") == "Q:001"), None)
if q100 is None or q001 is None:
    raise RuntimeError("Demo requires Q:100 and Q:001 in CORPORA['quran'].")

canon_text = " ".join(q100.verses)
print("\n--- Test 1: authentic Q:100 ---")
r1 = is_quranic(canon_text, "Q:100", canon_text=canon_text)
print(json.dumps({k: r1[k] for k in ("verdict", "K_fired", "fires", "ncd")}, indent=2, ensure_ascii=False))

print("\n--- Test 2: hand-crafted forgery (substitute one consonant in v1) ---")
v1 = q100.verses[0]
for i, ch in enumerate(v1):
    if ch in _enumerate.ARABIC_CONS_28:
        forged_v1 = v1[:i] + ("\u0628" if ch != "\u0628" else "\u062a") + v1[i+1:]
        break
else:
    raise RuntimeError("No consonant found in Q:100 v1.")
forged_text = " ".join([forged_v1] + list(q100.verses[1:]))
r2 = is_quranic(forged_text, "Q:100", canon_text=canon_text)
print(json.dumps({k: r2[k] for k in ("verdict", "K_fired", "fires", "ncd")}, indent=2, ensure_ascii=False))

print("\n--- Test 3: passing Q:001 under the Q:100 label (whole-surah mismatch) ---")
q001_text = " ".join(q001.verses)
r3 = is_quranic(q001_text, "Q:100", canon_text=canon_text)
print(json.dumps({k: r3[k] for k in ("verdict", "K_fired", "fires", "ncd")}, indent=2, ensure_ascii=False))

## Cell 10 — Persist receipt + CSVs + audit + self-check end

All outputs are written under `results/experiments/exp95e_full_114_consensus_universal/<scope>/`. Then the self-check runs in `end` mode: if any protected file changed during the run, it raises `IntegrityError`.

In [ ]:
from experiments._lib import self_check_end

per_surah_compact = {}
for lbl, row in per_surah.items():
    compact = {k: v for k, v in row.items()
               if k not in ("missed", "fires_K_atleast", "fires_solo", "ncd_min", "ncd_max")}
    compact["fires_K_atleast"] = list(row["fires_K_atleast"])
    compact["fires_solo"]      = dict(row["fires_solo"])
    compact["ncd_min"]         = dict(row["ncd_min"])
    compact["ncd_max"]         = dict(row["ncd_max"])
    per_surah_compact[lbl] = compact

elapsed_total = time.time() - t_score + elapsed_enum

prereg_fp = _audit.check_prereg_fingerprint(
    Path(_REPO_ROOT) / "experiments" / "exp95e_full_114_consensus_universal" / "PREREG.md",
    exp95e_run._PREREG_EXPECTED_HASH,
)

audit_report = _audit.collate_audit_report(
    tau_drift=tau_drift, prereg_fingerprint=prereg_fp,
    q100_regression=q100_reg, variant_counts=var_counts,
    missed_clusters=clusters,
    self_check_pre_ts=pre["ts_start"],
    runtime_seconds=round(elapsed_total, 2),
    scope=SCOPE,
)

report = {
    "experiment": exp95e_run.EXP,
    "schema_version": 1,
    "hypothesis": (
        "H37 — multi-compressor consensus K=2 across {gzip-9, bz2-9, lzma-preset-9, zstd-9} "
        "with τ frozen from exp95c closes single-letter forgery detection on every Quran "
        "surah at recall ≥ 0.999 aggregate, ≥ 0.99 per-surah."
    ),
    "prereg_document": "experiments/exp95e_full_114_consensus_universal/PREREG.md",
    "prereg_hash_actual":  prereg_fp.get("actual_hash"),
    "prereg_hash_expected": exp95e_run._PREREG_EXPECTED_HASH,
    "scope": SCOPE,
    "frozen_constants": {
        "gzip_level": exp95e_run.GZIP_LEVEL, "bz2_level": exp95e_run.BZ2_LEVEL,
        "lzma_preset": exp95e_run.LZMA_PRESET, "zstd_level": exp95e_run.ZSTD_LEVEL,
        "fpr_target": exp95e_run.FPR_TARGET, "headline_k": exp95e_run.HEADLINE_K,
        "protocol_drift_tol": exp95e_run.PROTOCOL_DRIFT_TOL,
        "exp94_baseline_gzip_recall": exp95e_run.EXP94_BASELINE_GZIP_RECALL,
    },
    "tau_per_compressor": receipt_tau,
    "tau_locked_snapshot": locked_tau,
    "ctrl_null": ctrl_null,
    "n_surahs_scored": len(per_surah_compact),
    "n_variants_total": agg["n_variants"],
    "aggregate": agg,
    "per_surah": per_surah_compact,
    "audit_report": audit_report,
    "verdict": verdict,
    "runtime_seconds": round(elapsed_total, 2),
}

outfile = out / f"{exp95e_run.EXP}.json"
with open(outfile, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=float)
exp95e_run._write_per_surah_csv(per_surah, out / "per_surah_table.csv")
exp95e_run._write_missed_csv(all_missed, out / "missed_variants.csv")
with open(out / "audit_report.json", "w", encoding="utf-8") as f:
    json.dump(audit_report, f, indent=2, ensure_ascii=False, default=float)

print(f"Wrote {outfile}")
print(f"Wrote {out / 'per_surah_table.csv'}")
print(f"Wrote {out / 'missed_variants.csv'}")
print(f"Wrote {out / 'audit_report.json'}")

self_check_end(pre, exp95e_run.EXP)
print(f"\nSelf-check PASSED — no protected file mutated during the run.")
print(f"\nVERDICT: {verdict}")
print(f"Runtime: {elapsed_total:.1f}s")

## Cell 11 — Package results into a zip with a one-click download button

Builds a single zip containing the receipt JSON, both CSVs, the audit report, and a `SUMMARY.txt` (so the zip is self-describing). Then displays a green download button below the cell — click it to save the zip to your computer's default download folder.

The button uses a base64-encoded data URI, so it works in Jupyter classic, JupyterLab, VS Code Jupyter, and Colab without any browser plugins. The zip is also written to disk under `results/experiments/.../<scope>/` if you prefer to grab it from there.

In [ ]:
import zipfile
import base64
from datetime import datetime
from IPython.display import display, HTML, FileLink

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f"exp95e_{SCOPE}_results_{ts}.zip"
zip_path = out / zip_name

# A self-describing summary that lives at the top of the zip
surahs_with_variants = sum(1 for r in per_surah.values() if r['n_variants'] > 0)
summary_lines = [
    "exp95e_full_114_consensus_universal — RUN SUMMARY",
    "=" * 64,
    f"Timestamp     : {ts}",
    f"Scope         : {SCOPE}",
    f"Workers       : {WORKERS}",
    f"Verdict       : {verdict}",
    f"Runtime       : {elapsed_total:.1f}s",
    f"",
    f"Surahs scored : {surahs_with_variants} / {len(per_surah)}",
    f"N variants    : {agg['n_variants']:,}",
    f"Aggregate K=1 : {agg['recall_K1']:.6f}",
    f"Aggregate K=2 : {agg['recall_K2']:.6f}  <-- HEADLINE",
    f"Aggregate K=3 : {agg['recall_K3']:.6f}",
    f"Aggregate K=4 : {agg['recall_K4']:.6f}",
    f"",
    f"ctrl K=1 FPR  : {ctrl_null['fpr_by_consensus_K']['1']:.4f}",
    f"ctrl K=2 FPR  : {ctrl_null['fpr_by_consensus_K']['2']:.4f}  (target <= {exp95e_run.FPR_TARGET})",
    f"ctrl K=3 FPR  : {ctrl_null['fpr_by_consensus_K']['3']:.4f}",
    f"ctrl K=4 FPR  : {ctrl_null['fpr_by_consensus_K']['4']:.4f}",
    f"",
    f"Q:100 K=2     : {q100_reg['K2_recall_q100']['actual']:.6f}  (expected 1.000000)",
    f"Q:100 gzip    : {q100_reg['gzip_solo_recall_q100']['actual']:.6f}  (expected {exp95e_run.EXP94_BASELINE_GZIP_RECALL:.6f})",
    f"",
    f"PREREG hash   : {prereg_fp.get('actual_hash', '?')}",
    f"PREREG ok     : {prereg_fp.get('ok')} (deferred={prereg_fp.get('deferred', False)})",
    f"τ drift max   : {tau_drift['max_drift']:.2e}",
    f"",
    "=" * 64,
    "FILES IN THIS ZIP",
    "=" * 64,
    "  exp95e_full_114_consensus_universal.json — main pre-registered receipt",
    "  per_surah_table.csv                     — one row per surah (recall, NCD min/max, etc.)",
    "  missed_variants.csv                     — every K_fired<2 variant with NCDs",
    "  audit_report.json                       — drift sentinels, fingerprints, clustering",
    "  SUMMARY.txt                             — this file",
    "",
]
summary_text = "\n".join(summary_lines)
print(summary_text)

artifacts = [
    "exp95e_full_114_consensus_universal.json",
    "per_surah_table.csv",
    "missed_variants.csv",
    "audit_report.json",
]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    zf.writestr("SUMMARY.txt", summary_text)
    for name in artifacts:
        f = out / name
        if f.exists():
            zf.write(f, arcname=name)

zip_size_kb = zip_path.stat().st_size / 1024
print(f"\nWrote {zip_path}")
print(f"Zip size: {zip_size_kb:.1f} KB")

# Read zip into base64 for the in-cell download button
with open(zip_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")

# Pick a colour based on verdict status
if verdict.startswith("PASS"):
    bg, border, hbg = "#e8f5e9", "#43a047", "#388e3c"
    icon = "\u2713"  # checkmark
elif verdict.startswith("PARTIAL"):
    bg, border, hbg = "#fff8e1", "#fbc02d", "#f57f17"
    icon = "\u26a0"  # warning
else:
    bg, border, hbg = "#ffebee", "#e53935", "#c62828"
    icon = "\u2717"  # cross

html = (
    f'<div style="padding:1em 1.2em; background:{bg}; border:2px solid {border}; '
    'border-radius:8px; margin-top:0.6em; font-family:sans-serif;">'
    f'<h3 style="margin:0 0 0.4em 0; color:{hbg};">{icon} Verdict: {verdict}</h3>'
    f'<p style="margin:0 0 0.5em 0; color:#333;">'
    f'Aggregate K=2 recall: <b>{agg["recall_K2"]:.6f}</b> &nbsp;'
    f'ctrl K=2 FPR: <b>{ctrl_null["fpr_by_consensus_K"]["2"]:.4f}</b> &nbsp;'
    f'Variants: <b>{agg["n_variants"]:,}</b> &nbsp;'
    f'Runtime: <b>{elapsed_total:.0f}s</b>'
    '</p>'
    f'<a download="{zip_name}" href="data:application/zip;base64,{b64}" '
    f'style="display:inline-block; padding:0.7em 1.4em; background:{hbg}; color:white; '
    'text-decoration:none; border-radius:4px; font-weight:bold; font-size:1.05em; '
    'box-shadow: 0 1px 3px rgba(0,0,0,0.15);">'
    f'\u2b07 Download {zip_name} ({zip_size_kb:.0f} KB)'
    '</a>'
    f'<p style="margin:0.7em 0 0 0; color:#555; font-size:0.88em;">'
    'The zip contains: receipt JSON + per-surah CSV + missed-variants CSV + audit report + SUMMARY.txt. '
    f'It is also saved to <code>{zip_path}</code> on disk.'
    '</p>'
    '</div>'
)
display(HTML(html))

# A second link that some Jupyter UIs render as a clickable file link
print("\nDirect file link (right-click \u2192 save as):")
display(FileLink(str(zip_path)))